# Vision Transformers (ViT), Image Embeddings & Visual Recognition

This interactive lab covers:
1. **An Image is Worth 16x16 Words**: Splitting 2D images into linear patch projections.
2. **Vision Transformer (ViT) Architecture**: `[CLS]` token, 1D position embeddings, and self-attention blocks.
3. **Dense Image Embeddings & Metric Learning**: Cosine similarity for visual search and retrieval.
4. **OCR & Sequence Vision Models**: Connecting patch representations to character sequence decoders.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('04-computer-vision/vision-transformers/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import torch
import torch.nn.functional as F
from vit_engine import (
    PatchEmbedding,
    VisionTransformerMini,
    compute_image_similarity,
    extract_l2_normalized_embedding,
)

print('Vision Transformer & Image Embedding engine loaded successfully!')

## 1. Patch Embedding: From 2D Pixels to 1D Tokens

Extracting non-overlapping $4 \times 4$ patches from an image of size $32 \times 32$ yields $(32/4)^2 = 64$ visual tokens.

In [ ]:
torch.manual_seed(42)
patch_embed = PatchEmbedding(in_channels=3, patch_size=4, embed_dim=64)

synthetic_img = torch.randn(2, 3, 32, 32)
patches = patch_embed(synthetic_img)

print(f'Input image batch shape:       {synthetic_img.shape}')
print(f'Flattened patch token sequence: {patches.shape}')
assert patches.shape == (2, 64, 64)

## 2. Full Vision Transformer Forward Pass

Inspecting the global visual representation aggregated at the `[CLS]` token.

In [ ]:
vit = VisionTransformerMini(
    img_size=32,
    patch_size=4,
    in_channels=3,
    num_classes=5,
    embed_dim=64,
    depth=3,
    num_heads=4
)

targets = torch.tensor([1, 4])
out = vit(synthetic_img, targets=targets)

print(f'Global [CLS] embedding shape: {out["cls_embedding"].shape}')
print(f'Local patch tokens shape:     {out["patch_tokens"].shape}')
print(f'Output class logits:          {out["logits"].shape}')
print(f'Cross-Entropy Loss:           {out["loss"].item():.4f}')

## 3. Metric Learning: Image Embeddings & Cosine Retrieval

Computing L2-normalized image embeddings on the unit hypersphere and evaluating pairwise similarity.

In [ ]:
# Extract normalized unit embeddings for 3 images
img_query = torch.randn(1, 3, 32, 32)
img_target_a = img_query + 0.05 * torch.randn(1, 3, 32, 32)  # Highly similar perturbation
img_target_b = torch.randn(1, 3, 32, 32)                    # Uncorrelated random image

emb_q = extract_l2_normalized_embedding(vit, img_query)
emb_a = extract_l2_normalized_embedding(vit, img_target_a)
emb_b = extract_l2_normalized_embedding(vit, img_target_b)

sim_qa = compute_image_similarity(emb_q, emb_a).item()
sim_qb = compute_image_similarity(emb_q, emb_b).item()

print(f'Cosine Similarity (Query, Similar A):      {sim_qa:.4f}')
print(f'Cosine Similarity (Query, Uncorrelated B): {sim_qb:.4f}')
assert sim_qa > sim_qb, 'Similar perturbation should yield higher cosine similarity!'